# Figure 9.11: Coefficient Shrinkage as λ Grows

Ridge keeps all coefficients but pulls them smoothly toward zero.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def draw(degree=6, noise=0.25, lam_max=40.0, seed=9):
    x, y, _, _ = poly_data(seed=seed, n=60, obs_noise=noise, gt_noise=0.0)
    lam_grid = np.linspace(0.0, lam_max, 80)
    coefs = np.stack([ridge_fit(x, y, degree, lam) for lam in lam_grid], axis=0)
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    for j in range(1, degree + 1):
        ax.plot(lam_grid, coefs[:, j], lw=2, label=fr"$\theta_{j}$")
    ax.set_title("Ridge coefficient paths")
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel("coefficient value")
    ax.grid(alpha=0.2)
    ax.legend(ncol=2, fontsize=9)
    plt.show()

widgets.interact(
    draw,
    degree=widgets.IntSlider(min=3, max=10, step=1, value=6, continuous_update=False),
    noise=widgets.FloatSlider(min=0.05, max=0.8, step=0.05, value=0.25, continuous_update=False),
    lam_max=widgets.FloatSlider(min=5.0, max=80.0, step=1.0, value=40.0, continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=9, continuous_update=False),
)
